# هدف ۲ — مقایسه‌ی سه مدل روی GPU خودِ Colab**بدون کلید API. بدون محدودیت نرخ.** مدل‌ها داخل Colab دانلود و روی کارت گرافیک اجرا می‌شوند.## پیش از شروع: GPU را روشن کن`Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save`## نکته‌ی فنی مهمهر سه مدل هم‌زمان در حافظه‌ی T4 (۱۵ گیگابایت) جا نمی‌شوند. این نوت‌بوک آن‌ها را **یکی‌یکی** بارگذاری، اجرا و از حافظه خارج می‌کند. اگر ترتیب سلول‌ها را به هم بزنی، خطای `CUDA out of memory` می‌گیری.

## ۱. بررسی GPU

In [ ]:
import torchif not torch.cuda.is_available():    raise SystemExit("✘ GPU روشن نیست → Runtime → Change runtime type → T4 GPU")print("✔ GPU:", torch.cuda.get_device_name(0))print(f"  حافظه: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} گیگابایت")

## ۲. نصب

In [ ]:
!pip -q install transformers accelerate bitsandbytes rouge-score bert-scoreprint("✔ نصب تمام شد")

## ۳. بارگذاری فایل‌ها`ctibench_core.py` و `all_attack_v1.csv` و `gold.json` را آپلود کن(آیکن پوشه 📁 در نوار کنار چپ).

In [ ]:
import osNEEDED=["ctibench_core.py","all_attack_v1.csv","gold.json"]if any(not os.path.exists(f) for f in NEEDED):    from google.colab import files; files.upload()for f in NEEDED: print(("  ✔ " if os.path.exists(f) else "  ✘ ")+f)

In [ ]:
import csv, json, syssys.path.insert(0,"."); csv.field_size_limit(10**9)import ctibench_core as Crows = list(csv.DictReader(open("all_attack_v1.csv", encoding="utf8")))GOLD = {g["idx"]: g["gold"] for g in json.load(open("gold.json", encoding="utf8"))}IDX  = sorted(GOLD)print(f"پیکره: {len(rows)} گزارش · برچسب طلایی: {len(GOLD)}")

## ۴. خط پایه — کفِ مقایسهروی CPU اجرا می‌شود و چند ثانیه طول می‌کشد. هر مدلی که از این اعداد جلو نزند، ارزش افزوده‌ای ندارد.انتظار: `lead3 ≈ 0.207` و `textrank ≈ 0.227` — اگر عدد دیگری دیدی یعنی داده درست بارگذاری نشده.

In [ ]:
from rouge_score import rouge_scorerimport pandas as pdscorer = rouge_scorer.RougeScorer(["rouge1","rouge2","rougeL"], use_stemmer=True)records = []for name, fn in C.BASELINES.items():    for i in IDX:        pred = fn(rows[i]["text"])        s = scorer.score(rows[i]["summary"], pred)        records.append(dict(model=name, idx=i, prediction=pred,                            rouge1=s["rouge1"].fmeasure, rouge2=s["rouge2"].fmeasure,                            rougel=s["rougeL"].fmeasure))print(pd.DataFrame(records).groupby("model")[["rouge1","rouge2","rougel"]].mean().round(3))

## ۵. انتخاب سه مدلمدل‌های زیر روی T4 رایگان جا می‌شوند. می‌توانی عوضشان کنی، ولی حواست باشد مدل بزرگ‌تر از حدود ۳ میلیارد پارامتر احتمالاً حافظه کم می‌آورد.> بعضی مدل‌ها (مثل Llama و Gemma) در HuggingFace نیازمند پذیرش شرایط‌اند. اگر خطای دسترسی گرفتی، یا در سایت HuggingFace شرایط را بپذیر و توکن بگذار، یا مدل را با یکی از جایگزین‌های آزاد عوض کن.

In [ ]:
MODELS = {    "qwen-1.5b": "Qwen/Qwen2.5-1.5B-Instruct",    "qwen-3b":   "Qwen/Qwen2.5-3B-Instruct",    "tinyllama": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",}LIMIT = 5              # با ۵ شروع کن؛ بعد روی len(IDX) بگذارMODES = ["zeroshot"]   # بعداً "fewshot" هم اضافه کنMAX_NEW = 320targets = IDX[:LIMIT]print(f"{len(MODELS)} مدل × {len(MODES)} راهبرد × {len(targets)} گزارش = "      f"{len(MODELS)*len(MODES)*len(targets)} اجرا")

## ۶. اجرای مدل‌ها — یکی‌یکیهر مدل بارگذاری، اجرا، و بلافاصله از حافظه خارج می‌شود. سلول را یک بار اجرا کن و صبر کن؛ اولین بار مدل دانلود می‌شود.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfigfrom tqdm.auto import tqdmimport gc, time# کوانتیزاسیون ۴ بیتی تا مدل در حافظه‌ی T4 جا شود.# در نسخه‌های جدید transformers باید صریح داده شود، نه با load_in_4bit=True.try:    BNB = BitsAndBytesConfig(load_in_4bit=True,                             bnb_4bit_compute_dtype=torch.float16,                             bnb_4bit_quant_type="nf4",                             bnb_4bit_use_double_quant=True)    QUANT = "4bit-nf4"except Exception as e:    BNB, QUANT = None, "fp16"    print("هشدار: کوانتیزاسیون در دسترس نیست، fp16 استفاده می‌شود —", e)def free_gpu():    gc.collect(); torch.cuda.empty_cache()for key, repo in MODELS.items():    print(f"\n━━━ بارگذاری {key} ({repo}) ━━━")    t0 = time.time()    tok = AutoTokenizer.from_pretrained(repo)    mdl = AutoModelForCausalLM.from_pretrained(repo, quantization_config=BNB,                                               device_map="auto", torch_dtype=torch.float16)    mdl.eval()    print(f"  بارگذاری در {time.time()-t0:.0f} ثانیه")    for mode in MODES:        for i in tqdm(targets, desc=f"{key}/{mode}", leave=False):            try:                prompt = C.build_prompt(mode, rows[i]["text"][:12000])                msgs = [{"role": "user", "content": prompt}]                text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)                ids  = tok([text], return_tensors="pt").to("cuda")                t1 = time.time()                with torch.no_grad():                    out = mdl.generate(**ids, max_new_tokens=MAX_NEW,                                       do_sample=False, pad_token_id=tok.eos_token_id)                pred = tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()                dt = time.time() - t1                ntok = out.shape[1] - ids["input_ids"].shape[1]                s = scorer.score(rows[i]["summary"], pred)                records.append(dict(model=key, mode=mode, idx=i, prediction=pred,                                    rouge1=s["rouge1"].fmeasure, rouge2=s["rouge2"].fmeasure,                                    rougel=s["rougeL"].fmeasure,                                    seconds=round(dt,2), tokens=ntok,                                    tok_per_sec=round(ntok/dt,1) if dt else None))            except Exception as e:                records.append(dict(model=key, mode=mode, idx=i, prediction="",                                    error=f"{type(e).__name__}: {str(e)[:150]}"))    del mdl, tok; free_gpu()    print(f"  ✔ {key} تمام شد و از حافظه خارج شد")df = pd.DataFrame(records)ok = df[df["error"].isna()] if "error" in df else dfprint("\n", ok.groupby("model")[["rouge1","rouge2","rougel"]].mean().round(3))

## ۷. کارایی محاسباتیهدف سوم پروپوزال به «تناسب برای محیط‌های با منابع محدود» اشاره دارد. سنجه‌ی درست **توکن بر ثانیه** است نه ثانیه‌ی مطلق، چون مدل‌ها طول خروجی متفاوتی تولید می‌کنند.

In [ ]:
if "tok_per_sec" in df:    perf = ok.groupby("model").agg(        میانگین_توکن_بر_ثانیه=("tok_per_sec","mean"),        میانگین_ثانیه=("seconds","mean"),        میانگین_توکن=("tokens","mean")).round(1)    print(perf.to_string())

## ۸. BERTScore`rescale_with_baseline=True` حتماً لازم است — بدون آن مقدار خام حتی برای دو متن بی‌ربط حدود ۰٫۸ است و قدرت تفکیک ندارد.

In [ ]:
from bert_score import score as bert_scorefree_gpu()mask = df["prediction"].astype(str).str.len() > 0preds = df.loc[mask,"prediction"].tolist()refs  = [rows[i]["summary"] for i in df.loc[mask,"idx"]]_,_,F1 = bert_score(preds, refs, lang="en", rescale_with_baseline=True, verbose=True)df.loc[mask,"bertscore"] = [float(x) for x in F1]print(df.groupby("model")[["rouge1","rougel","bertscore"]].mean().round(3))

## ۹. آزمون معناداری در برابر خط پایه

In [ ]:
from scipy.stats import wilcoxondef vs_baseline(model, baseline="textrank", metric="rougel"):    A = df[df["model"]==model].set_index("idx")[metric].dropna()    B = df[df["model"]==baseline].set_index("idx")[metric].dropna()    common = A.index.intersection(B.index)    if len(common) < 5:        return f"{model}: نمونه‌ی مشترک کم است ({len(common)})"    stat,p = wilcoxon(A[common], B[common])    d = A[common].mean() - B[common].mean()    if p >= 0.05:      v = "تفاوت معنادار نیست"    elif d > 0:        v = "بهتر از خط پایه ✔"    else:              v = "بدتر از خط پایه ✘"    return f"{model} ({A[common].mean():.3f}) vs {baseline} ({B[common].mean():.3f}) · n={len(common)} · p={p:.4f} → {v}"for m in MODELS:    print(" ", vs_baseline(m))

## ۱۰. ذخیره‌ی خروجی

In [ ]:
from datetime import datetimestamp = datetime.now().strftime("%Y%m%d-%H%M")df.to_csv(f"local-runs-{stamp}.csv", index=False)json.dump(dict(models=MODELS, modes=MODES, limit=LIMIT, max_new_tokens=MAX_NEW,               gpu=torch.cuda.get_device_name(0), quantization=QUANT,               decoding="greedy (do_sample=False)", timestamp=stamp),          open(f"local-meta-{stamp}.json","w"), ensure_ascii=False, indent=2)print("ذخیره شد:", f"local-runs-{stamp}.csv", f"local-meta-{stamp}.json", sep="\n  ")from google.colab import filesfiles.download(f"local-runs-{stamp}.csv")

---## در پایان‌نامه چه گزارش کنی۱. **نام دقیق مدل‌ها** و اینکه با کوانتیزاسیون ۴ بیتی اجرا شده‌اند — بدون این قید، اعداد به مدل کامل نسبت داده می‌شوند در حالی که متعلق به نسخه‌ی فشرده‌اند۲. **رمزگشایی حریصانه** (`do_sample=False`) — یعنی خروجی قطعی و بازتولیدپذیر است۳. **جدول اصلی همراه سطر خط پایه**۴. **توکن بر ثانیه** به‌عنوان سنجه‌ی کارایی، نه ثانیه‌ی مطلق۵. مقدار p هر مقایسه؛ اگر مدلی از خط پایه جلو نزد، صادقانه بنویس